# 02 - Camera-Local Zones

هذه المرحلة تحوّل مسارات Notebook 01 إلى أحداث مناطق لكل كاميرا بصورة مستقلة. الـHomography تستخدم لإسناد المنطقة فقط؛ لا يوجد Re-ID أو Global Fusion أو ربط هوية بين كاميرتين.


In [ ]:
from pathlib import Path
import json
import re

import cv2
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent

TABLES_DIR = PROJECT_ROOT / 'Output' / 'tables'
CONFIG_DIR = PROJECT_ROOT / 'Data' / 'config'
LOCAL_TRACKS_PATH = TABLES_DIR / 'local_tracks.csv'
ZONES_PATH = CONFIG_DIR / 'store_zones.json'
CALIBRATION_PATH = CONFIG_DIR / 'camera_calibration.generated.json'

def infer_store_id(camera_id):
    match = re.search(r'(place_\\d+)', str(camera_id), flags=re.IGNORECASE)
    return match.group(1).lower() if match else 'default_store'


def to_bool(values):
    return values.fillna(False).astype(str).str.strip().str.casefold().isin({'1', 'true', 'yes', 'y'})


def validate_homography(entry, camera_id):
    matrix = np.asarray(entry.get('homography'), dtype=np.float64)
    if matrix.shape != (3, 3) or not np.isfinite(matrix).all() or np.linalg.matrix_rank(matrix) < 3:
        raise ValueError(f'Invalid homography for {camera_id}')
    is_identity = np.allclose(matrix, np.eye(3), atol=1e-9)
    if is_identity and not bool(entry.get('reference_camera', False)):
        raise ValueError(f'Identity homography is allowed only for an explicit reference camera: {camera_id}')
    return matrix, 'reference_camera' if is_identity else 'configured'


def assign_zone(floor_x, floor_y, zones):
    point = (float(floor_x), float(floor_y))
    for zone in zones:
        polygon = np.asarray(zone['polygon'], dtype=np.float32)
        if cv2.pointPolygonTest(polygon, point, False) >= 0:
            return zone['zone_id'], zone['label_ar'], zone['kind']
    return 'outside', 'خارج المناطق', 'outside'


if not LOCAL_TRACKS_PATH.exists():
    raise FileNotFoundError('local_tracks.csv is missing. Run Notebook 01 first.')
if not CALIBRATION_PATH.exists():
    raise FileNotFoundError('camera_calibration.generated.json is missing. Zone assignment needs a calibration for every processed camera.')

local_tracks = pd.read_csv(LOCAL_TRACKS_PATH)
required_columns = {'camera_id', 'local_track_id', 'frame_index', 'timestamp_sec', 'foot_x', 'foot_y'}
missing_columns = sorted(required_columns.difference(local_tracks.columns))
if missing_columns:
    raise ValueError(f'local_tracks.csv is missing columns: {missing_columns}')
if local_tracks.empty:
    raise ValueError('local_tracks.csv is empty. Run Notebook 01 with readable camera videos.')

zones = json.loads(ZONES_PATH.read_text(encoding='utf-8'))
if not isinstance(zones, list) or not zones:
    raise ValueError('store_zones.json must contain at least one zone.')
for zone in zones:
    if not {'zone_id', 'label_ar', 'kind', 'polygon'}.issubset(zone):
        raise ValueError('Each zone requires zone_id, label_ar, kind, and polygon.')
    if len(zone['polygon']) < 3:
        raise ValueError('Zone ' + str(zone['zone_id']) + ' needs at least three polygon points.')

calibrations = json.loads(CALIBRATION_PATH.read_text(encoding='utf-8'))
events = local_tracks.copy()
events['camera_id'] = events['camera_id'].astype(str)
events['store_id'] = events['camera_id'].map(infer_store_id)
employee_values = events['is_employee'] if 'is_employee' in events.columns else pd.Series(False, index=events.index)
events['is_employee'] = to_bool(employee_values)
events['is_customer'] = ~events['is_employee']
events['camera_track_uid'] = (
    events['store_id'].astype(str) + '::' + events['camera_id'].astype(str) + '::' + events['local_track_id'].astype(str)
)

projected = pd.DataFrame(index=events.index, columns=['floor_x', 'floor_y'], dtype=float)
calibration_modes = pd.Series(index=events.index, dtype='object')
missing_calibrations = []

for camera_id, camera_rows in events.groupby('camera_id', sort=True):
    entry = calibrations.get(camera_id)
    if entry is None:
        missing_calibrations.append({'camera_id': camera_id, 'reason': 'missing_calibration'})
        continue
    try:
        homography, mode = validate_homography(entry, camera_id)
    except (TypeError, ValueError, np.linalg.LinAlgError) as error:
        missing_calibrations.append({'camera_id': camera_id, 'reason': str(error)})
        continue
    source_points = camera_rows[['foot_x', 'foot_y']].to_numpy(dtype=np.float32).reshape(-1, 1, 2)
    floor_points = cv2.perspectiveTransform(source_points, homography).reshape(-1, 2)
    projected.loc[camera_rows.index, ['floor_x', 'floor_y']] = floor_points
    calibration_modes.loc[camera_rows.index] = mode

missing_path = TABLES_DIR / 'missing_calibrations.csv'
pd.DataFrame(missing_calibrations, columns=['camera_id', 'reason']).to_csv(missing_path, index=False)
if missing_calibrations:
    cameras = ', '.join(item['camera_id'] for item in missing_calibrations)
    raise RuntimeError(f'Zone assignment stopped. Missing or invalid calibration for: {cameras}. Details were saved to {missing_path}.')

events['floor_x'] = projected['floor_x'].round(3)
events['floor_y'] = projected['floor_y'].round(3)
events['calibration_mode'] = calibration_modes
zone_values = [assign_zone(x, y, zones) for x, y in events[['floor_x', 'floor_y']].to_numpy()]
events[['zone_id', 'zone_label_ar', 'zone_kind']] = pd.DataFrame(zone_values, index=events.index)

preferred_columns = [
    'store_id', 'camera_id', 'camera_track_uid', 'local_track_id', 'is_employee', 'is_customer',
    'confidence', 'frame_index', 'timestamp_sec', 'foot_x', 'foot_y', 'x1', 'y1', 'x2', 'y2',
    'floor_x', 'floor_y', 'calibration_mode', 'zone_id', 'zone_label_ar', 'zone_kind',
]
zone_events = events[[column for column in preferred_columns if column in events.columns]].sort_values(
    ['store_id', 'camera_id', 'camera_track_uid', 'timestamp_sec', 'frame_index']
).reset_index(drop=True)
zone_events.to_csv(TABLES_DIR / 'zone_events.csv', index=False)

run_manifest = {
    'schema_version': 1,
    'identity_scope': 'camera_local',
    'identity_note': 'camera_track_uid never links a person across cameras',
    'camera_ids': sorted(zone_events['camera_id'].unique().tolist()),
    'zone_event_rows': int(len(zone_events)),
}
(TABLES_DIR / 'camera_zone_run.json').write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')
print(f'Zone events: {len(zone_events):,}')
print(f'Camera-local tracks: {zone_events.camera_track_uid.nunique():,}')
print('Saved:', (TABLES_DIR / 'zone_events.csv').relative_to(PROJECT_ROOT))
zone_events.head()


شغّل Notebook 01 أولًا. هذه المرحلة تحتاج local_tracks.csv ومعايرة صالحة لكل كاميرا تدخل التشغيل.

المخرجات القابلة للمراجعة: zone_events.csv وcamera_zone_run.json وmissing_calibrations.csv. لا تنتج هذه المرحلة Global IDs أو أي نتائج cross-camera.
